<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 12 — Graphs Reference

A lookup guide. Notebooks 01 and 02 are meant to be read in order; this one is
meant to be **searched**. Keep it open while you work on Homework 11.

Every section runs on the small route map defined in the first cell: the same
eleven airports as the test network of Homework 11. Run the setup cell, then
jump to the section you need.

| Section | When you need it |
|---|---|
| [A — The words](#A) | node, edge, degree, path, component, ... |
| [B — Writing a graph down](#B) | edge list, adjacency list, which to use |
| [C — Building a graph from edges](#C) | directed, undirected, weighted, every node a key |
| [D — Asking a graph](#D) | neighbours, degree, is there an edge, hubs |
| [E — Breadth-first search](#E) | hops to everything; the loop and its three rules |
| [F — A shortest path](#F) | `came_from`, walking back |
| [G — Reachability and components](#G) | what hangs together |
| [H — Depth-first search](#H) | the other order, and what it is for |
| [I — Weighted edges](#I) | path length; Dijkstra with `heapq` |
| [J — The same in networkx](#J) | the library, function by function |
| [K — Cost](#K) | how the work grows |
| [L — Symptoms and causes](#L) | `KeyError`, endless loops, wrong distances |

**In Homework 11 you write Sections C to G yourself, without networkx.** The
library calls in Section J are for checking your answers in the notebook.

In [ ]:
import heapq
from collections import deque

import networkx as nx

# (source, target, km).  Most routes are listed in both directions; the one to
# ANC is one way, and HNL and OGG are cut off from the rest.
EDGES = [("ATL", "MIA", 950), ("MIA", "ATL", 950), ("ATL", "CHI", 950), ("CHI", "ATL", 950),
         ("ATL", "NYC", 1200), ("NYC", "ATL", 1200), ("ATL", "LAX", 3100), ("LAX", "ATL", 3100),
         ("CHI", "DEN", 1450), ("DEN", "CHI", 1450), ("CHI", "NYC", 1150), ("NYC", "CHI", 1150),
         ("NYC", "BOS", 300), ("BOS", "NYC", 300), ("DEN", "LAX", 1350), ("LAX", "DEN", 1350),
         ("DEN", "SEA", 1650), ("SEA", "DEN", 1650), ("LAX", "SEA", 1550), ("SEA", "LAX", 1550),
         ("BOS", "SEA", 4000), ("SEA", "BOS", 4000), ("HNL", "OGG", 160), ("OGG", "HNL", 160),
         ("SEA", "ANC", 2300)]
print(len(EDGES), "edges")

---

<a id="A"></a>
## A — The words

| Word | Means |
|---|---|
| **graph** (network) | a set of nodes and a set of edges between pairs of them |
| **node** (vertex) | one of the things |
| **edge** (link) | a connection between two nodes |
| **directed** edge | leads one way, from its source to its target |
| **undirected** edge | leads both ways |
| **weight** | a number on an edge: a distance, a cost, a time |
| **neighbour** | a node one edge away |
| **degree** | the number of edges at a node; in a directed graph, *out-degree* (leaving) and *in-degree* (arriving) |
| **path** | a sequence of nodes, each joined to the next by an edge |
| **hop** | one edge of a path; a path of n nodes has n − 1 hops |
| **length** of a path | its number of hops, or the sum of its weights: say which |
| **shortest path** | a path of smallest length between two nodes; often not unique |
| **distance** | the length of a shortest path |
| **reachable** | there is a path from one node to the other |
| **connected component** | a set of nodes that can all reach each other, with no edge leading out |
| **cycle** | a path that returns to where it began |
| **hub** | a node of unusually high degree |
| **diameter** | the largest distance between any two nodes |
| **sparse** | far fewer edges than pairs of nodes; nearly all real networks |

---

<a id="B"></a>
## B — Writing a graph down

| Form | Looks like | Good for | Poor for |
|---|---|---|---|
| **edge list** | `[("ATL", "MIA", 950), ...]` | files; going through all edges | "who are X's neighbours?" (reads every edge) |
| **adjacency list** | `{"ATL": {"MIA": 950, ...}, ...}` | neighbours, degree, is there an edge: one step each | "all edges" needs a double loop |
| **adjacency matrix** | a table with a row and a column per node | dense graphs; matrix arithmetic | memory: n × n cells, nearly all empty |

The usual pipeline: read an edge list from a file, build an adjacency list,
ask the adjacency list.

Which adjacency list?

| Neighbours kept in | `graph[node]` is | Use when |
|---|---|---|
| a **set** | `{"MIA", "CHI"}` | unweighted; "is there an edge" in one step |
| a **dictionary** | `{"MIA": 950, "CHI": 950}` | weighted: the value is the weight |
| a **list** | `["MIA", "CHI"]` | you need the order of the file, or repeated edges |

---

<a id="C"></a>
## C — Building a graph from edges

In [ ]:
def build_graph(edges, directed=True):
    """Return {node: {neighbour: weight}} with EVERY node as a key."""
    graph = {}
    for source, target, weight in edges:
        if source not in graph:
            graph[source] = {}
        if target not in graph:                 # also when nothing leaves it
            graph[target] = {}
        if target not in graph[source] or weight < graph[source][target]:
            graph[source][target] = weight
        if not directed:
            if source not in graph[target] or weight < graph[target][source]:
                graph[target][source] = weight
    return graph


graph = build_graph(EDGES)
both_ways = build_graph(EDGES, directed=False)

for node in sorted(graph):
    print(node, graph[node])

| You want | Write this |
|---|---|
| an unweighted graph | `graph.setdefault(a, set()).add(b)`, and `graph.setdefault(b, set())` |
| a weighted graph | `graph.setdefault(a, {})[b] = weight`, and `graph.setdefault(b, {})` |
| undirected | add the edge in both directions |
| every node present | create the entry for the target as well as the source |
| nodes that have no edges at all | add them first: `graph = {node: {} for node in nodes}` |
| the number of nodes | `len(graph)` |
| the number of edges, directed | `sum(len(found) for found in graph.values())` |
| the number of edges, undirected | the same, divided by 2 |
| all edges back out | `[(a, b, w) for a in graph for b, w in graph[a].items()]` |
| the graph with every edge reversed | build it from `(b, a, w)` |

Reading an edge list from a CSV file with a header:

```python
import csv

edges = []
with open(path, encoding="utf-8", newline="") as handle:
    for row in csv.DictReader(handle):
        edges.append((row["source"].strip(), row["target"].strip(), float(row["km"])))
```

Everything a CSV reader returns is text; convert the weight.

In [ ]:
print(len(graph), "nodes;", sum(len(found) for found in graph.values()), "directed edges;",
      sum(len(found) for found in both_ways.values()) // 2, "undirected")
print("ANC:", graph["ANC"], " <- a key with no neighbours")

---

<a id="D"></a>
## D — Asking a graph

| You want | Write this | Careful |
|---|---|---|
| is the node there? | `node in graph` | |
| its neighbours | `graph[node]` | `KeyError` for an unknown node |
| the same, safely | `graph.get(node, {})` | |
| in a fixed order | `sorted(graph[node])` | |
| its degree | `len(graph[node])` | out-degree in a directed graph |
| is there an edge a → b? | `a in graph and b in graph[a]` | check `a` first |
| the weight of an edge | `graph[a][b]` | |
| in-degree of a node | `sum(1 for a in graph if node in graph[a])` | reads the whole graph |
| the nodes by degree | `sorted(graph, key=lambda n: (-len(graph[n]), n))` | ties by name |
| nodes nothing leaves | `[n for n in graph if not graph[n]]` | |
| the average degree | `sum(len(f) for f in graph.values()) / len(graph)` | |

`graph.setdefault(node, {})` and `graph[node]` on a `defaultdict` both **add**
the node when it is missing. Asking must not change the graph: use `in` or
`.get`.

In [ ]:
print(sorted(graph["SEA"]), len(graph["SEA"]))
print("SEA -> ANC:", "ANC" in graph["SEA"], "  ANC -> SEA:", "SEA" in graph["ANC"])
print(sorted(graph, key=lambda node: (-len(graph[node]), node))[:4])
print("arriving at ANC:", sum(1 for node in graph if "ANC" in graph[node]))

---

<a id="E"></a>
## E — Breadth-first search

In [ ]:
def hop_distances(graph, start):
    """Return {node: fewest hops from start} for every reachable node."""
    if start not in graph:
        return {}
    hops = {start: 0}                       # found, and how far
    queue = deque([start])                  # found, not yet expanded
    while queue:
        node = queue.popleft()              # 1. from the FRONT
        for neighbour in sorted(graph[node]):
            if neighbour not in hops:       # 2. skip what was seen
                hops[neighbour] = hops[node] + 1    # 3. note it on entering the queue
                queue.append(neighbour)
    return hops


print(hop_distances(graph, "ATL"))
print(hop_distances(graph, "ANC"))

| Rule | Break it and |
|---|---|
| take from the **front** (`popleft`) | distances are wrong: the search runs deep before it runs wide |
| **skip** nodes already found | it never ends on a graph with a cycle |
| note a node when it **enters** the queue | nodes are queued many times; slow, and wrong if you overwrite |

| You want | Write this |
|---|---|
| a queue | `queue = deque([start])`; `queue.append(x)`; `queue.popleft()` |
| is it empty? | `while queue:` |
| the hops to one node | `hop_distances(graph, start).get(target)` (None when unreachable) |
| all nodes exactly k hops away | `[n for n, h in hops.items() if h == k]` |
| how many nodes per number of hops | a tally of `hops.values()` |
| the farthest node | `max(hops, key=hops.get)` |
| stop at a maximum depth | `if hops[node] == limit: continue` before the inner loop |

Why it gives *shortest* hop counts: nodes leave the queue in order of
distance, so the first time a node is found is by a route with the fewest
edges. Only true when every edge counts the same.

---

<a id="F"></a>
## F — A shortest path

In [ ]:
def shortest_path(graph, start, target):
    """Return a path with the fewest hops, or None."""
    if start not in graph or target not in graph:
        return None
    came_from = {start: None}
    queue = deque([start])
    while queue:
        node = queue.popleft()
        if node == target:
            break                           # optional: nothing nearer is left
        for neighbour in sorted(graph[node]):
            if neighbour not in came_from:
                came_from[neighbour] = node
                queue.append(neighbour)
    if target not in came_from:
        return None

    path = []
    node = target
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()
    return path


print(shortest_path(graph, "MIA", "ANC"))
print(shortest_path(graph, "ANC", "MIA"), shortest_path(graph, "ATL", "ATL"))

| Case | Result |
|---|---|
| a path exists | the list of nodes from start to target |
| start is the target | `[start]` |
| no path | `None` |
| a node is not in the graph | `None` |

- `came_from` is both the record of what has been seen and the notes to walk
  back along. The start's note is `None`, which is where the walk stops.
- The walk produces the path backwards: `reverse()` it.
- Hops of a path: `len(path) - 1`.
- **Shortest paths are not unique.** Which one you get depends on the order
  the neighbours are looked at. To check a path, test that it is valid and
  has the right length:

```python
def is_path(graph, path, start, target):
    return (path[0] == start and path[-1] == target
            and all(path[i + 1] in graph[path[i]] for i in range(len(path) - 1)))
```

---

<a id="G"></a>
## G — Reachability and components

In [ ]:
def reachable(graph, start):
    return set(hop_distances(graph, start))


def components(graph):
    """The connected parts of an UNDIRECTED graph, largest first."""
    seen = set()
    parts = []
    for node in graph:
        if node not in seen:
            part = reachable(graph, node)
            seen |= part
            parts.append(part)
    parts.sort(key=lambda part: (-len(part), min(part)))
    return parts


print(sorted(reachable(graph, "ATL")))
print(sorted(reachable(graph, "ANC")))
print(components(both_ways))

| You want | Write this |
|---|---|
| can `b` be reached from `a`? | `b in reachable(graph, a)` |
| nodes that cannot be reached | `set(graph) - reachable(graph, a)` |
| is the whole graph connected? | `len(components(graph)) == 1` |
| the largest component | `components(graph)[0]` |
| which component a node is in | build `{node: number}` while collecting the parts |
| nodes that can reach `a` | search from `a` in the graph with every edge reversed |

Components as defined here are for **undirected** graphs. For a directed
graph, build it with both directions first to ask which nodes hang together
at all. (Requiring that every node can reach every other *along the arrows*
gives "strongly connected components", a harder question.)

---

<a id="H"></a>
## H — Depth-first search

Take from the **back** of the queue and the search follows one route as far
as it goes before trying another. The container is then called a *stack*.

In [ ]:
def depth_first_order(graph, start):
    """The order in which a depth-first search visits the nodes."""
    seen = set()
    order = []
    stack = [start]
    while stack:
        node = stack.pop()                  # from the BACK
        if node in seen:
            continue
        seen.add(node)
        order.append(node)
        for neighbour in sorted(graph[node], reverse=True):
            if neighbour not in seen:
                stack.append(neighbour)
    return order


print("depth-first:  ", depth_first_order(graph, "ATL"))
print("breadth-first:", list(hop_distances(graph, "ATL")))

| | Breadth-first | Depth-first |
|---|---|---|
| container | queue: first in, first out | stack: last in, first out |
| order | by distance from the start | one branch to its end, then back up |
| finds every reachable node | yes | yes |
| finds shortest hop counts | **yes** | no |
| good for | shortest paths, "how far" | reachability, cycles, ordering tasks by their requirements |

For reachability and components either works. For anything about *distance*,
only breadth-first.

---

<a id="I"></a>
## I — Weighted edges

In [ ]:
def path_length(graph, path):
    """The sum of the weights along a path; None if a step is not an edge."""
    if not path:
        return None
    total = 0.0
    for position in range(len(path) - 1):
        a, b = path[position], path[position + 1]
        if a not in graph or b not in graph[a]:
            return None
        total += graph[a][b]
    return total


def fewest_km_path(graph, start, target):
    """Dijkstra: the path with the smallest total weight, or None."""
    if start not in graph or target not in graph:
        return None
    best = {start: 0.0}
    came_from = {start: None}
    queue = [(0.0, start)]                          # a priority queue: (so far, node)
    done = set()
    while queue:
        so_far, node = heapq.heappop(queue)         # the NEAREST unfinished node
        if node in done:
            continue
        done.add(node)
        if node == target:
            break
        for neighbour in sorted(graph[node]):
            length = so_far + graph[node][neighbour]
            if neighbour not in best or length < best[neighbour]:
                best[neighbour] = length            # a shorter way was found
                came_from[neighbour] = node
                heapq.heappush(queue, (length, neighbour))
    if target not in done:
        return None
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = came_from[node]
    return path[::-1]


by_hops = shortest_path(graph, "MIA", "SEA")
by_km = fewest_km_path(graph, "MIA", "SEA")
print(by_hops, len(by_hops) - 1, "hops,", path_length(graph, by_hops), "km")
print(by_km, len(by_km) - 1, "hops,", path_length(graph, by_km), "km")

| | Breadth-first search | Dijkstra |
|---|---|---|
| question | fewest edges | smallest total weight |
| takes next | the oldest node in the queue | the nearest node in the priority queue |
| container | `deque` | `heapq` |
| a node's distance | final when first found | may improve until the node is taken out |
| needs | nothing | weights that are **not negative** |
| with all weights equal | — | gives the same answer as breadth-first search |

`heapq` in three lines:

```python
heapq.heappush(queue, (priority, item))     # add
priority, item = heapq.heappop(queue)       # remove and return the SMALLEST
queue = []                                  # an ordinary list holds the heap
```

Tuples are compared by their first item, so `(distance, node)` comes out by
distance.

**What "shortest" means is a modelling decision**, made by what you put on
the edges:

| On every edge | A shortest path is |
|---|---|
| 1 (or nothing) | the route with the fewest steps |
| kilometres | the shortest distance |
| minutes | the fastest |
| the ticket price | the cheapest |
| minutes, plus a penalty for each change | a compromise |

---

<a id="J"></a>
## J — The same in networkx

| You want | networkx |
|---|---|
| an empty undirected / directed graph | `nx.Graph()`, `nx.DiGraph()` |
| add edges | `G.add_edges_from([(a, b), ...])` |
| add weighted edges | `G.add_weighted_edges_from([(a, b, w), ...], weight="km")` |
| add a node without edges | `G.add_node(x)` |
| from a table | `nx.from_pandas_edgelist(table, "source", "target", edge_attr="km", create_using=nx.DiGraph)` |
| numbers of nodes and edges | `G.number_of_nodes()`, `G.number_of_edges()` |
| neighbours | `list(G[x])`, `list(G.successors(x))` in a `DiGraph` |
| degree | `G.degree(x)`; `G.out_degree(x)`, `G.in_degree(x)` |
| is there an edge? | `G.has_edge(a, b)` |
| the weight of an edge | `G[a][b]["km"]` |
| hops to everything | `nx.single_source_shortest_path_length(G, start)` |
| a path with the fewest hops | `nx.shortest_path(G, a, b)` |
| its number of hops | `nx.shortest_path_length(G, a, b)` |
| a path with the smallest weight | `nx.shortest_path(G, a, b, weight="km")` |
| that weight | `nx.shortest_path_length(G, a, b, weight="km")` |
| the weight of a given path | `nx.path_weight(G, path, weight="km")` |
| all shortest paths | `nx.all_shortest_paths(G, a, b)` |
| is there a path? | `nx.has_path(G, a, b)` |
| everything reachable | `nx.descendants(G, a)` (without `a` itself) |
| components (undirected) | `nx.connected_components(G)` |
| ignore the directions | `G.to_undirected()` |
| a picture of a small graph | `nx.draw_networkx(G, ax=ax)` |

In [ ]:
G = nx.DiGraph()
G.add_weighted_edges_from(EDGES, weight="km")

assert dict(nx.single_source_shortest_path_length(G, "ATL")) == hop_distances(graph, "ATL")
assert nx.shortest_path_length(G, "MIA", "SEA") == len(by_hops) - 1
assert nx.shortest_path(G, "MIA", "SEA", weight="km") == by_km
assert nx.path_weight(G, by_km, weight="km") == path_length(graph, by_km)
assert sorted(map(sorted, nx.connected_components(G.to_undirected()))) == sorted(
    map(sorted, components(both_ways)))
assert not nx.has_path(G, "ANC", "SEA") and shortest_path(graph, "ANC", "SEA") is None
print("the hand-written functions and networkx agree")

- `nx.shortest_path` raises `NetworkXNoPath` when there is none, and
  `NodeNotFound` for an unknown node, where the functions above return `None`.
  Ask `nx.has_path` first, or catch the exception.
- Without `weight=...` networkx counts hops, even when the edges carry
  weights. Forgetting the argument is the usual mistake.
- networkx is in the course environment on the cluster. It is **not** in the
  homework pipeline: do not import it in a submission.

---

<a id="K"></a>
## K — Cost

For a graph with n nodes and m edges:

| Operation | Edge list | Adjacency list |
|---|---|---|
| neighbours of a node | m | 1 (then one step per neighbour) |
| is there an edge a → b? | m | 1 |
| build it | — | m |
| breadth-first search from one node | n × m | n + m |
| a shortest path between two nodes | | n + m |
| components | | n + m |
| Dijkstra | | about (n + m) × log n |
| distances between **all** pairs | | n × (n + m) |
| memory | m | n + m |
| memory of an adjacency matrix | | n × n |

- One search costs about as much as reading the graph once.
- "For every pair" or "from every node" multiplies that by n. On 3,000
  airports it is seconds; on a network of a million nodes it is days.
- `queue.pop(0)` on a list costs a step per item in the list; `popleft()` on
  a `deque` costs one.

---

<a id="L"></a>
## L — Symptoms and causes

| Symptom | Likely cause | Fix |
|---|---|---|
| `KeyError: 'ANC'` during a search | a node that is only ever a target has no entry | create entries for targets when building (Section C) |
| the graph has fewer nodes than expected | the same | the same |
| the search never ends | nodes already found are not skipped | `if neighbour not in hops` |
| distances too large for some nodes | taking from the back of the queue (`pop()`) | `popleft()` |
| the same node appears many times in the queue | noting nodes when they leave, not when they enter | note on entering |
| the path comes out backwards | walked `came_from` from the target | `reverse()` |
| the path is missing its first node | the walk stops at the start instead of after it | walk until the note is `None` |
| `None` for a node to itself | no special case, or an early return | `[start]` |
| a path exists on paper, the function says `None` | a one-way edge; or the graph was built directed and should not be | check `directed` |
| the graph grows while you ask it | `setdefault` or a `defaultdict` adding missing nodes | `in`, `.get` |
| components come out as 1 where 2 are expected, or the reverse | components asked of a directed graph | build with both directions |
| weights are concatenated: `"950950"` | weights left as text from the CSV | `float(...)` |
| "shortest" path is very long in km | breadth-first search counts hops | Dijkstra, or `weight=` in networkx |
| `TypeError: '<' not supported` in `heapq` | two entries with equal priority and items that cannot be compared | put a tie-breaker in the tuple: `(priority, name)` |
| a correct path is marked wrong by a check | it was compared with one particular shortest path | check validity and length |
| works on the cluster, fails in the pipeline | `import networkx` in the submission | standard library only |